# RL Basics and MDPs — Exercises

Companion to the note [RL Basics and MDPs](RL%20Basics%20and%20MDPs.md).

Practise the MDP formalism $(S,A,P,R,\gamma)$: returns, the Bellman expectation and optimality equations, dynamic programming (policy evaluation, value iteration, policy iteration), Monte Carlo evaluation and the exploration-exploitation tradeoff in bandits. By hand first, then from scratch in NumPy.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

rng = np.random.default_rng(0)
# A small random MDP used by the code exercises.
# P[s, a, s'] = transition probability, R[s, a] = expected immediate reward.
S, nA = 5, 2
P = rng.dirichlet(np.ones(S), size=(S, nA))
R = rng.normal(size=(S, nA))
gamma = 0.9

def exact_policy_value(P, R, pi, gamma):
    """Reference: solve the Bellman expectation equation as a linear system. pi[s, a] = pi(a|s)."""
    P_pi = np.einsum('sa,sat->st', pi, P)
    r_pi = (pi * R).sum(1)
    return np.linalg.solve(np.eye(len(r_pi)) - gamma * P_pi, r_pi)

## Part A · Concepts

### Exercise 1 · Is it an MDP?
*🧠 Concept · ★☆☆*

For each problem, say whether the stated observation is a **Markov state** (i.e. $P(s'\mid s,a)$ depends only on the current state).
If not, propose a state that is.

1. Chess, state = current board position (plus whose turn, castling rights).
2. An Atari game (Pong), state = a single screen frame.
3. A thermostat, state = current room temperature, where the heater takes 10 minutes to warm up.
4. Poker, state = your own cards and the public cards.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: would knowing the *history* help predict the next state beyond what the current observation tells you?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Markov**: the board (with turn, castling and en-passant info) determines the future completely.
2. **Not Markov**: a single frame shows the ball's position but not its velocity. Stacking the last 4 frames (as in DQN) gives an approximately Markov state.
3. **Not Markov**: the effect of an action depends on how long the heater has been on. Add the heater's on-time / recent actions to the state.
4. **Not Markov** for you (partially observable): opponents' hidden cards and their betting history matter. This is a POMDP; a belief state over hidden cards (built from the history) is Markov.

</details>

### Exercise 2 · Exploration vs exploitation
*🧠 Concept · ★☆☆*

The note lists three exploration strategies: **ε-greedy**, **UCB** and **Thompson sampling**.
Describe in one sentence each how they choose an action, and say which one(s) explore *uniformly at random*
and which explore in a *directed* way (towards uncertain actions).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

UCB adds a bonus that shrinks as an action is tried more often; Thompson samples from a posterior over each action's value.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **ε-greedy**: with probability $\varepsilon$ take a uniformly random action, otherwise the action with the highest estimate. Exploration is *undirected*: a clearly bad action is tried as often as a promising but uncertain one.
- **UCB**: pick $\arg\max_a \hat Q(a) + c\sqrt{\ln t / N(a)}$; rarely tried actions get a large bonus ("optimism in the face of uncertainty"). *Directed*.
- **Thompson sampling**: keep a posterior over each action's mean reward, draw one sample per action, act greedily w.r.t. the samples. Actions are chosen with the probability that they are optimal. *Directed* and Bayesian.

</details>

### Exercise 3 · Taxonomy: planning, model-free, on/off-policy
*🧠 Concept · ★★☆*

Classify each method as (i) **planning with a known model** or **learning from experience**, (ii) **model-based** or **model-free**,
and, where it applies, (iii) **on-policy** or **off-policy**:

value iteration · Monte Carlo evaluation · SARSA · Q-learning · Dyna-Q (learns $P$ and $R$ from data, then plans with them).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Off-policy means the policy being *learned about* (target) differs from the one *generating the data* (behaviour). See [[Q-Learning and Policy Gradients]].

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Method | Planning / learning | Model | On/off-policy |
|---|---|---|---|
| Value iteration | planning (needs $P,R$) | model-based | n/a (no data) |
| Monte Carlo | learning | model-free | on-policy in its basic form (off-policy with importance sampling) |
| SARSA | learning | model-free | **on**-policy: target uses the action $a'$ actually taken |
| Q-learning | learning | model-free | **off**-policy: target uses $\max_{a'}$ (greedy policy) while behaving ε-greedily |
| Dyna-Q | learning **and** planning | model-based (learned model) | off-policy (Q-learning updates) |

</details>

### Exercise 4 · The discount factor
*🧠 Concept · ★★☆*

Give three reasons for using $\gamma<1$. Then explain the rule of thumb that $\gamma$ corresponds to an
**effective horizon** of about $1/(1-\gamma)$ steps. What horizon do $\gamma = 0.9$ and $\gamma=0.99$ give?
What happens to an agent with $\gamma=0$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Consider $\sum_{k\ge0}\gamma^k$ and how quickly $\gamma^k$ decays.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Reasons: (1) the infinite sum $G_t$ stays finite for bounded rewards ($|G_t|\le R_{\max}/(1-\gamma)$); (2) it makes the Bellman operator a
$\gamma$-contraction, so DP converges; (3) it models uncertainty about the future / preference for earlier reward (and in finance, interest).

Since $\sum_k \gamma^k = 1/(1-\gamma)$, a constant reward stream is worth as much as $1/(1-\gamma)$ undiscounted steps; also $\gamma^k$ has decayed to
$e^{-1}$ after roughly $1/(1-\gamma)$ steps. $\gamma=0.9 \Rightarrow 10$ steps, $\gamma = 0.99 \Rightarrow 100$ steps.
With $\gamma=0$ the agent is **myopic**: it maximises only the immediate reward $r_{t+1}$.

</details>

## Part B · By hand

### Exercise 5 · Computing a return
*✍️ By hand · ★☆☆*

An episode yields rewards $r_1, r_2, r_3, r_4 = 1, 0, 2, 3$ and then terminates. With $\gamma = 0.9$, compute $G_0$ and $G_2$.
Verify $G_0 = r_1 + \gamma G_1$.

In [ ]:
G0 = None
G2 = None

_r = np.array([1, 0, 2, 3.]); _g = 0.9 ** np.arange(4)
check('G0', G0, _r @ _g)
check('G2', G2, _r[2:] @ _g[:2])

<details>
<summary><b>💡 Hint</b></summary>

$G_t = \sum_k \gamma^k r_{t+k+1}$. For $G_2$ only $r_3, r_4$ remain.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$G_0 = 1 + 0.9\cdot 0 + 0.81\cdot 2 + 0.729\cdot 3 = 1 + 1.62 + 2.187 = 4.807$.
$G_2 = 2 + 0.9\cdot 3 = 4.7$, $G_1 = 0 + 0.9\cdot 4.7 = 4.23$, and $r_1 + \gamma G_1 = 1 + 0.9\cdot 4.23 = 4.807$ ✔.

```python
G0 = 4.807
G2 = 4.7
```

</details>

### Exercise 6 · Infinite constant reward
*✍️ By hand · ★☆☆*

An agent receives reward $+1$ at every step forever. Compute its return for $\gamma=0.95$. Then the reward is $+1$ for
the first 10 steps and $0$ afterwards: what is the return now (same $\gamma$)?

In [ ]:
G_inf = None
G_10 = None

check('G infinite', G_inf, 1/(1-0.95))
check('G 10 steps', G_10, sum(0.95**k for k in range(10)))

<details>
<summary><b>💡 Hint</b></summary>

Geometric series: $\sum_{k=0}^{n-1}\gamma^k = \frac{1-\gamma^n}{1-\gamma}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$G_\infty = \frac{1}{1-0.95} = 20$. $G_{10} = \frac{1-0.95^{10}}{0.05} = \frac{1-0.5987}{0.05} \approx 8.025$.
So the first 10 steps already account for 40% of the infinite return.

```python
G_inf = 20.0
G_10 = (1 - 0.95**10) / 0.05
```

</details>

### Exercise 7 · Bellman expectation as a linear system
*✍️ By hand · ★★☆*

Under a fixed policy $\pi$, a two-state chain has state-to-state transitions and expected rewards

$$P_\pi = \begin{pmatrix}0.5 & 0.5\\ 0.2 & 0.8\end{pmatrix},\qquad r_\pi = \begin{pmatrix}1\\0\end{pmatrix},\qquad \gamma = 0.9.$$

Write the Bellman expectation equation in matrix form $V = r_\pi + \gamma P_\pi V$, solve it **by hand** for $V^\pi(s_1), V^\pi(s_2)$.

In [ ]:
V_hand = None  # [V(s1), V(s2)]

check('V^pi', V_hand, np.linalg.solve(np.eye(2) - 0.9*np.array([[0.5,0.5],[0.2,0.8]]), [1, 0]), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$V = (I-\gamma P_\pi)^{-1} r_\pi$. Compute $I - 0.9P_\pi$ first.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$I-0.9P_\pi = \begin{pmatrix}0.55 & -0.45\\ -0.18 & 0.28\end{pmatrix}$, determinant $0.073$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$I-\gamma P_\pi = \begin{pmatrix}0.55 & -0.45\\ -0.18 & 0.28\end{pmatrix}$, $\det = 0.154 - 0.081 = 0.073$.
$(I-\gamma P_\pi)^{-1} r_\pi = \frac{1}{0.073}\begin{pmatrix}0.28 & 0.45\\ 0.18 & 0.55\end{pmatrix}\begin{pmatrix}1\\0\end{pmatrix}
= \begin{pmatrix}0.28/0.073\\ 0.18/0.073\end{pmatrix} \approx \begin{pmatrix}3.836\\ 2.466\end{pmatrix}$.

$s_1$ is worth more because it pays the reward immediately; $s_2$ pays nothing and is "sticky" (stays with prob. 0.8), so reaching the reward from there takes longer and is discounted more.
Direct solve costs $O(|S|^3)$, which is why iterative policy evaluation is used for large $S$.

```python
V_hand = [0.28 / 0.073, 0.18 / 0.073]
```

</details>

### Exercise 8 · Value iteration by hand: the myopic trap
*✍️ By hand · ★★☆*

Deterministic MDP with states $A, B$, $\gamma=0.9$:

| state | action | reward | next state |
|---|---|---|---|
| A | stay | 1 | A |
| A | go | 0 | B |
| B | stay | 2 | B |
| B | go | 0 | A |

Starting from $V_0 = 0$, perform two sweeps of value iteration $V_{k+1}(s) = \max_a [R(s,a) + \gamma V_k(s')]$ to get $V_2$.
Then find $V^*(A)$ and $V^*(B)$ exactly. Does the greedy action in $A$ w.r.t. $V_2$ agree with the optimal one?

In [ ]:
V2 = None      # [V2(A), V2(B)]
V_star = None  # [V*(A), V*(B)]

def _vi(n):
    V = np.zeros(2)
    for _ in range(n):
        V = np.array([max(1 + 0.9*V[0], 0.9*V[1]), max(2 + 0.9*V[1], 0.9*V[0])])
    return V
check('V2', V2, _vi(2))
check('V*', V_star, _vi(2000))

<details>
<summary><b>💡 Hint 1</b></summary>

$V_1 = (1, 2)$. For $V_2(A)$ compare $1 + 0.9\cdot V_1(A)$ with $0 + 0.9\cdot V_1(B)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For $V^*$: staying in $B$ forever is worth $2/(1-\gamma)$. From $A$, either stay forever or move once to $B$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$V_1 = (\max(1,0), \max(2,0)) = (1, 2)$.
$V_2(A) = \max(1 + 0.9\cdot1,\ 0.9\cdot 2) = \max(1.9, 1.8) = 1.9$; $V_2(B) = \max(2 + 1.8,\ 0.9) = 3.8$.

$V^*(B) = 2/(1-0.9) = 20$ (stay forever). $V^*(A) = \max(1/(1-0.9),\ 0 + 0.9\cdot 20) = \max(10, 18) = 18$: **go** to $B$.

W.r.t. $V_2$ the greedy action in $A$ is "stay" (1.9 > 1.8) but the optimal action is "go": after only two sweeps the information
about $B$'s long-term value has not propagated. From $k=3$ on, $V_k(A)$ prefers "go" ($V_3(A)=\max(2.71, 3.42)$).

```python
V2 = [1.9, 3.8]
V_star = [18.0, 20.0]
```

</details>

### Exercise 9 · From V* to Q*
*✍️ By hand · ★★☆*

Using $Q^*(s,a) = \sum_{s'}P(s'\mid s,a)[R(s,a) + \gamma V^*(s')]$ and the MDP of the previous exercise,
compute all four values $Q^*(A,\text{stay}), Q^*(A,\text{go}), Q^*(B,\text{stay}), Q^*(B,\text{go})$.
Check that $V^*(s) = \max_a Q^*(s,a)$. Why is $Q^*$ more convenient than $V^*$ for a model-free agent?

In [ ]:
Q_star = None  # [[Q(A,stay), Q(A,go)], [Q(B,stay), Q(B,go)]]

check('Q*', Q_star, [[1 + 0.9*18, 0.9*20], [2 + 0.9*20, 0.9*18]])

<details>
<summary><b>💡 Hint</b></summary>

Use $V^*(A) = 18$, $V^*(B)=20$. Each action leads to one next state with probability 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Q^*(A,\text{stay}) = 1 + 0.9\cdot 18 = 17.2$, $Q^*(A,\text{go}) = 0 + 0.9\cdot 20 = 18$,
$Q^*(B,\text{stay}) = 2 + 0.9\cdot 20 = 20$, $Q^*(B,\text{go}) = 0 + 0.9\cdot 18 = 16.2$.
Maxima: $18 = V^*(A)$, $20 = V^*(B)$ ✔.

With $Q^*$ the optimal action is just $\arg\max_a Q^*(s,a)$: no model $P$ is needed to act greedily. With only $V^*$
you need a one-step lookahead through $P$ and $R$. This is why model-free control learns $Q$ ([[Q-Learning and Policy Gradients]]).

```python
Q_star = [[17.2, 18.0], [20.0, 16.2]]
```

</details>

### Exercise 10 · Contraction and the number of sweeps
*✍️ By hand · ★★★*

The Bellman optimality operator $(TV)(s) = \max_a\sum_{s'}P(s'\mid s,a)[R + \gamma V(s')]$ satisfies
$\lVert TV - TU\rVert_\infty \le \gamma\lVert V - U\rVert_\infty$.

(a) Prove this. (b) Rewards lie in $[0,1]$ and $\gamma = 0.9$; you start value iteration at $V_0 = 0$. Using
$\lVert V_k - V^*\rVert_\infty \le \gamma^k\lVert V_0 - V^*\rVert_\infty$, how many sweeps $k$ guarantee an error $\le 0.01$?

In [ ]:
k_sweeps = None

check('sweeps', k_sweeps, int(np.ceil(np.log(0.01 / 10) / np.log(0.9))))

<details>
<summary><b>💡 Hint 1</b></summary>

Use $|\max_a f(a) - \max_a g(a)| \le \max_a |f(a) - g(a)|$ and that $\sum_{s'} P(s'\mid s,a) = 1$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

With rewards in $[0,1]$, $0 \le V^* \le 1/(1-\gamma) = 10$, so the initial error is at most 10.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) For each $s$:
$|(TV)(s) - (TU)(s)| \le \max_a \big|\gamma\sum_{s'}P(s'\mid s,a)(V(s') - U(s'))\big| \le \gamma\max_a\sum_{s'}P(s'\mid s,a)\lVert V-U\rVert_\infty = \gamma\lVert V-U\rVert_\infty$.
Taking the max over $s$ gives the claim. By Banach's fixed-point theorem $T$ has a unique fixed point $V^*$ and VI converges to it.

(b) Need $0.9^k\cdot 10 \le 0.01 \iff k \ge \frac{\ln(10^{-3})}{\ln 0.9} = 65.56$, so $k = 66$ sweeps.

```python
k_sweeps = 66
```

</details>

## Part C · Dynamic programming in NumPy

### Exercise 11 · Iterative policy evaluation
*💻 Code · ★☆☆*

Implement `policy_evaluation(P, R, pi, gamma, tol)` that repeatedly applies the Bellman expectation backup
$V(s) \leftarrow \sum_a\pi(a\mid s)\big[R(s,a) + \gamma\sum_{s'}P(s'\mid s,a)V(s')\big]$ until the max change is below `tol`.
Evaluate the uniform random policy on the setup MDP (`P`, `R`, `gamma`).

In [ ]:
def policy_evaluation(P, R, pi, gamma, tol=1e-10):
    # TODO: V = zeros(S); loop: Q = R + gamma * P @ V  (shape S x nA); V_new = (pi * Q).sum(1)
    return None

pi_unif = np.full((S, nA), 1 / nA)
V_unif = policy_evaluation(P, R, pi_unif, gamma)

check('policy evaluation', V_unif, exact_policy_value(P, R, pi_unif, gamma))

<details>
<summary><b>💡 Hint</b></summary>

`P @ V` with `P` of shape `(S, nA, S)` and `V` of shape `(S,)` gives the expected next value for each $(s,a)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each sweep is a $\gamma$-contraction, so the error shrinks by at least a factor $\gamma$ per sweep. The result matches the direct
linear solve $V = (I-\gamma P_\pi)^{-1}r_\pi$ from Exercise 7.

```python
def policy_evaluation(P, R, pi, gamma, tol=1e-10):
    V = np.zeros(P.shape[0])
    while True:
        Q = R + gamma * P @ V
        V_new = (pi * Q).sum(1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new

pi_unif = np.full((S, nA), 1 / nA)
V_unif = policy_evaluation(P, R, pi_unif, gamma)
```

</details>

### Exercise 12 · Value iteration
*💻 Code · ★★☆*

Implement `value_iteration(P, R, gamma, tol)` returning `(V, policy)` where `policy[s]` is the greedy action.
Verify against **brute force**: with 5 states and 2 actions there are only $2^5 = 32$ deterministic policies, so you can
evaluate all of them exactly and take the best. Store your results in `V_vi`, `policy_vi`.

In [ ]:
def value_iteration(P, R, gamma, tol=1e-10):
    # TODO
    return None, None

V_vi, policy_vi = value_iteration(P, R, gamma)

import itertools
_best = max((exact_policy_value(P, R, np.eye(nA)[list(p)], gamma) for p in itertools.product(range(nA), repeat=S)),
            key=lambda v: v.sum())
check('V* (vs brute force)', V_vi, _best)
check('greedy policy is optimal', None if policy_vi is None else exact_policy_value(P, R, np.eye(nA)[policy_vi], gamma), _best)

<details>
<summary><b>💡 Hint</b></summary>

Backup: `Q = R + gamma * P @ V`; `V_new = Q.max(1)`. After convergence, `policy = Q.argmax(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

An optimal deterministic policy dominates all others in **every** state simultaneously, so the brute-force maximiser (by sum) has
$V^{\pi}=V^*$ elementwise. Value iteration finds it without enumerating $|A|^{|S|}$ policies.

```python
def value_iteration(P, R, gamma, tol=1e-10):
    V = np.zeros(P.shape[0])
    while True:
        Q = R + gamma * P @ V
        V_new = Q.max(1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, Q.argmax(1)
        V = V_new

V_vi, policy_vi = value_iteration(P, R, gamma)
```

</details>

### Exercise 13 · Policy iteration
*💻 Code · ★★☆*

Implement `policy_iteration(P, R, gamma)`: start with policy all zeros; alternate **exact** policy evaluation
(linear solve) and **greedy improvement** until the policy no longer changes. Return `(V, policy, n_iter)`.
Compare with value iteration: same policy? How many iterations did it take?

In [ ]:
def policy_iteration(P, R, gamma):
    # TODO
    return None, None, None

V_pi, policy_pi, n_iter_pi = policy_iteration(P, R, gamma)

check('PI value = V* (brute force from the previous exercise)', V_pi, _best)

<details>
<summary><b>💡 Hint</b></summary>

Use `exact_policy_value(P, R, np.eye(nA)[policy], gamma)` for evaluation, then `policy_new = (R + gamma * P @ V).argmax(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Policy iteration typically converges in a handful of iterations (here 2–3) because each improvement step is a full greedy step w.r.t. the
exact value; the policy improvement theorem guarantees $V^{\pi_{k+1}}\ge V^{\pi_k}$ and there are finitely many policies.
Each iteration is more expensive ($O(|S|^3)$ solve) than a VI sweep.

```python
def policy_iteration(P, R, gamma):
    policy = np.zeros(P.shape[0], dtype=int)
    for it in range(1, 1000):
        V = exact_policy_value(P, R, np.eye(P.shape[1])[policy], gamma)
        new = (R + gamma * P @ V).argmax(1)
        if np.array_equal(new, policy):
            return V, policy, it
        policy = new

V_pi, policy_pi, n_iter_pi = policy_iteration(P, R, gamma)
print('iterations:', n_iter_pi, 'policy:', policy_pi)
```

</details>

### Exercise 14 · Monte Carlo policy evaluation
*💻 Code · ★★★*

Pretend you do not know `P`: you can only **sample** transitions with `step(s, a)` (given below).
Estimate $V^\pi$ of the uniform random policy by averaging sampled returns: for each start state run 300 rollouts of length
$H = 60$ (truncation error $\le\gamma^{60}R_{\max}/(1-\gamma)$ is tiny) and average the discounted sum of rewards.
Store the estimate in `V_mc` and compare with `V_unif`.

In [ ]:
mc_rng = np.random.default_rng(1)
def step(s, a):
    """Sample (reward, next_state). Reward is R[s,a] plus a bit of noise."""
    s_next = mc_rng.choice(S, p=P[s, a])
    return R[s, a] + 0.1 * mc_rng.normal(), s_next

def mc_evaluate(n_rollouts=300, H=60):
    # TODO: return an array of shape (S,)
    return None

V_mc = mc_evaluate()

check('MC estimate close to exact', V_mc, exact_policy_value(P, R, np.full((S, nA), 0.5), gamma), tol=0.35)

<details>
<summary><b>💡 Hint</b></summary>

Inside a rollout: `a = mc_rng.integers(nA)`, `r, s = step(s, a)`, `G += gamma**t * r`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MC is unbiased (up to truncation) but has high variance: the error shrinks like $1/\sqrt{n}$. TD learning
([[Q-Learning and Policy Gradients]]) bootstraps from $V(s')$ instead of waiting for the full return, trading some bias for much lower variance.

```python
mc_rng = np.random.default_rng(1)
def step(s, a):
    s_next = mc_rng.choice(S, p=P[s, a])
    return R[s, a] + 0.1 * mc_rng.normal(), s_next

def mc_evaluate(n_rollouts=300, H=60):
    V = np.zeros(S)
    disc = gamma ** np.arange(H)
    for s0 in range(S):
        total = 0.0
        for _ in range(n_rollouts):
            s, rs = s0, np.empty(H)
            for t in range(H):
                rs[t], s = step(s, mc_rng.integers(nA))
            total += rs @ disc
        V[s0] = total / n_rollouts
    return V

V_mc = mc_evaluate()
print(np.round(V_mc, 2), np.round(exact_policy_value(P, R, np.full((S, nA), 0.5), gamma), 2))
```

</details>

## Part D · Exploration in bandits

### Exercise 15 · ε-greedy vs UCB on a 10-armed bandit
*💻 Code · ★★☆*

A 10-armed Bernoulli bandit has success probabilities `p_arms` (given). Implement `run_bandit(strategy, T)` for
`strategy in {'greedy', 'eps', 'ucb'}` with incremental mean estimates $\hat Q(a) \leftarrow \hat Q(a) + \frac{1}{N(a)}(r - \hat Q(a))$:
- `'greedy'`: always $\arg\max \hat Q$ (after trying each arm once),
- `'eps'`: ε-greedy with $\varepsilon=0.1$,
- `'ucb'`: $\arg\max_a \hat Q(a) + \sqrt{2\ln t / N(a)}$ (after trying each arm once).

Return the **fraction of steps on which the best arm was pulled**. Average over 20 seeds with $T=2000$ and store the results in the dict `frac_best`.

In [ ]:
p_arms = np.array([0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.75])

def run_bandit(strategy, T, seed):
    r = np.random.default_rng(seed)
    # TODO: return fraction of pulls of arm argmax(p_arms)
    return None

frac_best = None  # {'greedy': ..., 'eps': ..., 'ucb': ...}

if frac_best is not None:
    check('UCB and eps beat pure greedy', min(frac_best['ucb'], frac_best['eps']) > frac_best['greedy'], True)
else:
    check('UCB and eps beat pure greedy', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Pull each arm once first so that $N(a)\ge1$. For ε-greedy: `if r.random() < 0.1: a = r.integers(10)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Pure greedy often locks onto a mediocre arm that got lucky early. ε-greedy keeps exploring, but wastes 10% of pulls on
arms that are clearly bad forever. UCB's bonus shrinks as $N(a)$ grows, so exploration concentrates on arms that *could* still be best;
its regret grows only logarithmically in $T$. With these seeds: greedy ≈ 0.49, ε-greedy ≈ 0.78, UCB ≈ 0.67.
UCB with $c=\sqrt2$ is conservative at small $T$ (it keeps checking the 0.55 arm), but ε-greedy's fraction is capped at
$0.9 + 0.1/10 = 0.91$ forever, whereas UCB's fraction tends to 1 as $T	o\infty$ (try $T = 20000$).

```python
p_arms = np.array([0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.75])

def run_bandit(strategy, T, seed):
    r = np.random.default_rng(seed)
    K = len(p_arms); Q = np.zeros(K); N = np.zeros(K); best = 0
    for t in range(1, T + 1):
        if t <= K:
            a = t - 1
        elif strategy == 'eps' and r.random() < 0.1:
            a = r.integers(K)
        elif strategy == 'ucb':
            a = np.argmax(Q + np.sqrt(2 * np.log(t) / N))
        else:
            a = np.argmax(Q)
        rew = float(r.random() < p_arms[a])
        N[a] += 1; Q[a] += (rew - Q[a]) / N[a]
        best += a == np.argmax(p_arms)
    return best / T

frac_best = {s: np.mean([run_bandit(s, 2000, seed) for seed in range(20)]) for s in ('greedy', 'eps', 'ucb')}
print(frac_best)
```

</details>

### Exercise 16 · Thompson sampling
*💻 Code · ★★★*

Add a `'thompson'` strategy for the same bandit: keep a $\text{Beta}(\alpha_a, \beta_a)$ posterior per arm (start at $\text{Beta}(1,1)$),
sample $\theta_a\sim\text{Beta}(\alpha_a,\beta_a)$, pull $\arg\max_a\theta_a$, then update $\alpha_a \mathrel{+}= r$, $\beta_a \mathrel{+}= 1-r$.
Store the average fraction of best-arm pulls (20 seeds, $T=2000$) in `frac_ts`. How does it compare with ε-greedy?

In [ ]:
def run_thompson(T, seed):
    r = np.random.default_rng(seed)
    # TODO
    return None

frac_ts = None

if frac_ts is not None:
    check('Thompson beats eps-greedy', frac_ts > np.mean([run_bandit('eps', 2000, s) for s in range(20)]), True)
else:
    check('Thompson beats eps-greedy', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`theta = r.beta(alpha, beta)` draws one sample per arm at once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Thompson sampling pulls each arm with the posterior probability that it is the best. Once the posterior of a bad arm is concentrated
below the best arm's, that arm is essentially never pulled again, unlike ε-greedy. Here it pulls the best arm ≈ 92% of the time, more than ε-greedy (≈ 78%) and UCB (≈ 67%); it is typically on par with or better than UCB in practice.

```python
def run_thompson(T, seed):
    r = np.random.default_rng(seed)
    K = len(p_arms); al = np.ones(K); be = np.ones(K); best = 0
    for t in range(T):
        a = np.argmax(r.beta(al, be))
        rew = float(r.random() < p_arms[a])
        al[a] += rew; be[a] += 1 - rew
        best += a == np.argmax(p_arms)
    return best / T

frac_ts = np.mean([run_thompson(2000, seed) for seed in range(20)])
print('Thompson:', frac_ts)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [RL Basics and MDPs](RL%20Basics%20and%20MDPs.md).*